### In questa esercitazione alleneremo un transformers decoder only su un dataset composto da dialoghi e corrispondenti riassunti


In [1]:
import pandas as pd
import re

In [2]:
def get_train_test_data(data_dir):
    # Get the train data
    train_data = pd.read_json(f"{data_dir}/train.json")
    train_data.drop(['id'], axis=1, inplace=True)

    # Get the test data
    test_data = pd.read_json(f"{data_dir}/test.json")
    test_data.drop(['id'], axis=1, inplace=True)

    return train_data, test_data
train_dataset, test_dataset= get_train_test_data('./data')
print(len(train_dataset))
train_dataset.head(5)

14732


,summary,dialogue
0,Amanda baked cookies and will bring Jerry some...,Amanda: I baked cookies. Do you want some?\r\...
1,Olivia and Olivier are voting for liberals in ...,Olivia: Who are you voting for in this electio...
2,Kim may try the pomodoro technique recommended...,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa..."
3,Edward thinks he is in love with Bella. Rachel...,"Edward: Rachel, I think I'm in ove with Bella...."
4,"Sam is confused, because he overheard Rick com...",Sam: hey overheard rick say something\r\nSam:...


In [3]:
#Creiamo il tokenizer
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
# Inizializza un tokenizer BPE

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Configura il trainer
trainer = BpeTrainer(
    vocab_size=5000, # Per un dataset piccolo come il nostro 10 mila è sufficiente
    min_frequency=2,
    special_tokens=["[PAD]", "[BOS]", "[EOS]", "[UNK]"]
)

texts = train_dataset.summary.to_list() + train_dataset.dialogue.to_list() + test_dataset.summary.to_list() + test_dataset.dialogue.to_list()
# Avvia l'addestramento sui tuoi dati
tokenizer.train_from_iterator(texts, trainer)

# Salva il tokenizer per non doverlo rifare
tokenizer.save("tokenizer_summarization.json")

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTSummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]

        # Prompt: Tutto ciò che diamo al modello come contesto
        prompt_text = f"[BOS] {item['dialogue']} \n [EOS]:"
        # Target: Quello che il modello deve imparare a scrivere
        summary_text = f" {item['summary']} [EOS]"

        # 2. Tokenizziamo senza aggiungere token speciali automatici (li abbiamo messi noi a mano)
        prompt_ids = self.tokenizer.encode(prompt_text).ids
        summary_ids = self.tokenizer.encode(summary_text).ids

        # 3. Concatenazione per l'INPUT (Il modello vede tutto)
        full_input_ids = prompt_ids + summary_ids

        # 4. Creazione delle LABELS (Copia dell'input)
        labels = list(full_input_ids)

        # 5. APPLICAZIONE DEL MASKING (-100)
        # Sostituiamo tutti gli ID corrispondenti al prompt con -100 (-100 sarà l'index che daremo alla loss CrossEntropy come parametro in ignore_index. Cosìn facendo quando l'algoritmo di backprop incontra questo indice non calcola la loss o meglio la setta a zero. Cosi facendo il modello non spreca tempo ad "impararare" il dialogo e i padding tokens
        prompt_len = len(prompt_ids)
        for i in range(prompt_len):
            labels[i] = -100

        # 6. Padding e Truncation
        # Se è troppo lungo, tagliamo
        if len(full_input_ids) > self.max_len:
            full_input_ids = full_input_ids[:self.max_len]
            labels = labels[:self.max_len]
        else:
            # Se è troppo corto, aggiungiamo padding
            pad_len = self.max_len - len(full_input_ids)
            full_input_ids = full_input_ids + [0] * pad_len # 0 è [PAD]
            labels = labels + [-100] * pad_len # Anche il padding va ignorato nella loss

        return {
            "input_ids": torch.tensor(full_input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

train_dataset_py = GPTSummarizationDataset(data=train_dataset,  tokenizer=tokenizer, max_len=512)
test_dataset_py = GPTSummarizationDataset(data=test_dataset, tokenizer=tokenizer, max_len=512)


In [5]:
#Definisco i dataloader
BATCH_SIZE = 8 # Regola in base alla memoria della  GPU

train_loader = DataLoader(
    train_dataset_py,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset_py,
    batch_size=BATCH_SIZE,
    shuffle=True
)
# Testiamo un batch
example_batch = next(iter(train_loader))

print(f"Shape Input: {example_batch['input_ids'].shape}") # [8, 512]
print(f"Shape Label: {example_batch['labels'].shape}") # [8, 128]

Shape Input: torch.Size([8, 512])
Shape Label: torch.Size([8, 512])


### Adesso definiamo il nostro transfomer decoder only

In [6]:
from torch import nn

class GPTLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Solo Self-Attention Causale
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [7]:
### Definiamo il positional encoding
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Creiamo una matrice di zeri (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # Vettore delle posizioni (0, 1, 2, ..., max_len)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calcoliamo il termine di divisione (il denominatore della formula)
        # Usiamo il logspace per stabilità numerica
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Applichiamo il seno alle posizioni pari e il coseno a quelle dispari
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Aggiungiamo una dimensione per il batch (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        # register_buffer indica a PyTorch che questo non è un parametro da allenare
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        # Sommiamo il positional encoding all'embedding del testo
        x = x + self.pe[:, :x.size(1), :]
        return x

In [8]:
import torch
import torch.nn as nn
import math

class GPT(nn.Module):
    def __init__(self,
                 vocab_size,
                 d_model,
                 n_layers,
                 n_heads,
                 d_ff,
                 max_len,
                 dropout,
                 pad_idx):
        super().__init__()

        self.d_model = d_model
        self.pad_idx = pad_idx

        # 1. Embedding e Positional Encoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        # 2. La pila di GPTLayer (quelli che mi hai incollato tu)
        self.layers = nn.ModuleList([
            GPTLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # 3. Normalizzazione Finale (Standard nei modelli GPT prima dell'output)
        self.norm_final = nn.LayerNorm(d_model)

        # 4. Proiezione sul vocabolario (Testa del modello)
        self.fc_out = nn.Linear(d_model, vocab_size)



    def make_causal_mask(self, x):
        # x shape: [batch_size, seq_len]
        seq_len = x.shape[1]

        # 1. Maschera Causale (Triangolare)
        # Crea una matrice piena di "True" sopra la diagonale (futuro)
        # device=x.device assicura che la maschera sia sulla GPU se i dati sono su GPU
        causal_mask = torch.triu(torch.ones((seq_len, seq_len), device=x.device), diagonal=1).bool()


        return causal_mask

    def forward(self, x):
        # x shape: [batch_size, seq_len]

        # 1. Creiamo la maschera causale (il modello non deve vedere il futuro)
        mask = self.make_causal_mask(x)

        # 2. Embedding + Positional Encoding
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x) # Gestisce internamente la lunghezza variabile
        x = self.dropout(x)

        # 3. Passaggio attraverso i layer
        for layer in self.layers:
            x = layer(x, mask)

        # 4. Normalizzazione e Output
        x = self.norm_final(x)
        logits = self.fc_out(x)

        return logits

In [9]:
# Parametri
VOCAB_SIZE = tokenizer.get_vocab_size()
D_MODEL = 256
N_LAYERS = 4
N_HEADS = 8
D_FF = 512
MAX_LEN = 1024 # Deve contenere Dialogo + Riassunto
DROPOUT = 0.1
PAD_IDX = tokenizer.token_to_id("[PAD]")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Istanziazione del modello GPT
model = GPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(device)

print(f"Modello GPT creato con {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parametri.")

Modello GPT creato con 4,673,928 parametri.


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
from tqdm import tqdm

# Configurazione Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Iperparametri di Training
LEARNING_RATE = 0.0005
EPOCHS = 5
CLIP = 1.0 # Soglia per il gradient clipping

# Definiamo PAD_IDX

# Inizializziamo il modello
model = model.to(device)

# Ottimizzatore: AdamW è il migliore per i Transformer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

# Loss Function: Ignora gli indici di Padding
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# Scheduler: Se la loss di validazione non migliora per 2 epoche, dimezza il LR
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [21]:
def train_step_gpt(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    # Barra di progresso
    progress_bar = tqdm(iterator, desc="Training GPT", leave=False)

    for batch in progress_bar:
        # 1. Carichiamo i dati
        # Shape: [Batch_Size, Seq_Len]
        input_ids = batch['input_ids'].to(device)


        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device) # Contiene i -100 sul dialogo

        # 2. Shifting dei dati (Next Token Prediction)
        # L'Input del modello è tutto tranne l'ultimo token
        # Esempio: "A B C" -> deve predire "B C D"
        x = input_ids[:, :-1]

        # Il Target è tutto tranne il primo token
        y = labels[:, 1:]

        optimizer.zero_grad()

        # 3. Forward Pass
        # Passiamo x.
        # Output Shape: [Batch_Size, Seq_Len - 1, Vocab_Size]
        output = model(x)

        # 4. Calcolo Loss
        # Appiattiamo: (Batch * (Seq_Len-1), Vocab_Size) vs (Batch * (Seq_Len-1))
        loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))

        # 5. Backward e Ottimizzazione
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    return epoch_loss / len(iterator)

def evaluate_step_gpt(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for batch in tqdm(iterator, desc="Evaluating GPT", leave=False):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # Stesso shifting del training
            x = input_ids[:, :-1]
            y = labels[:, 1:]

            output = model(x)

            loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [22]:
best_val_loss = float('inf')
save_path = 'best_transformer_decoder_only.pt'

print(f"--- Inizio Training per {5} epoche ---")

for epoch in range(EPOCHS):
    start_time = time.time()

    # --- FASE DI TRAINING ---
    train_loss = train_step_gpt(model, train_loader, optimizer, criterion, CLIP)

    # --- FASE DI VALIDAZIONE ---
    val_loss = evaluate_step_gpt(model, test_loader, criterion)

    # --- SCHEDULER STEP ---
    # Controlla se deve abbassare il Learning Rate
    scheduler.step(val_loss)

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    # --- SALVATAGGIO MODELLO ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        save_msg = f"-> Modello salvato in {save_path}"
    else:
        save_msg = ""

    # --- LOGGING ---
    print(f'Epoch: {epoch+1:02} | Time: {int(epoch_mins)}m {int(epoch_secs)}s')
    print(f'\tTrain Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    print(f'\t{save_msg}')
    print('-' * 60)

--- Inizio Training per 5 epoche ---


Epoch: 01 | Time: 1m 13s
	Train Loss: 2.8559 | Val Loss: 3.5992
	-> Modello salvato in best_transformer_decoder_only.pt
------------------------------------------------------------


Epoch: 02 | Time: 1m 15s
	Train Loss: 2.7033 | Val Loss: 3.5905
	-> Modello salvato in best_transformer_decoder_only.pt
------------------------------------------------------------


Epoch: 03 | Time: 1m 10s
	Train Loss: 2.6511 | Val Loss: 3.5968
	
------------------------------------------------------------


Epoch: 04 | Time: 1m 10s
	Train Loss: 2.6137 | Val Loss: 3.6203
	
------------------------------------------------------------


Epoch: 05 | Time: 1m 10s
	Train Loss: 2.5879 | Val Loss: 3.6390
	
------------------------------------------------------------


In [43]:
def ask_gpt(model, tokenizer, dialogue, device, temperature = 0.7, repetition_penalty = 1.6,  max_new_tokens=50):
    model.eval()

    # 1.
    # Il modello si aspetta questa struttura per capire che deve riassumere
    prompt = f"[BOS] {dialogue} \n [EOS]:"

    # 2. Tokenizzazione
    input_ids = tokenizer.encode(prompt).ids

    # Trasformiamo in tensore e aggiungiamo la dimensione del Batch (da [N] a [1, N])
    input_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)

    print("--- Inizio Generazione ---")

    # 3. Ciclo di Generazione (Autoregressivo)
    for _ in range(max_new_tokens):
        # Forward pass (senza calcolare gradienti)
        with torch.no_grad():
            outputs = model(input_tensor)

        # Prendiamo i logits dell'ULTIMO token generato
        # Shape: [1, Seq_Len, Vocab] -> ci interessa l'ultima fetta temporale
        next_token_logits = outputs[:, -1, :]

        # Divide i logits per la temperatura.
        # Temp bassa (<1) rende i picchi più alti (più sicuro).
        # Temp alta (>1) appiattisce la curva (più casuale).
        next_token_logits = next_token_logits / temperature

        # Prendiamo tutti gli ID che sono già nell'input corrente
        current_context = input_tensor[0].tolist()

        # Usiamo set() per non penalizzare due volte la stessa parola
        for token_id in set(current_context):
            # Se il logit è positivo, lo dividiamo (lo abbassiamo)
            if next_token_logits[0, token_id] > 0:
                next_token_logits[0, token_id] /= repetition_penalty
            # Se il logit è negativo, lo moltiplichiamo (lo rendiamo più negativo)
            else:
                next_token_logits[0, token_id] *= repetition_penalty

        # Greedy Search: Scegliamo il token con probabilità più alta
        predicted_id = torch.argmax(next_token_logits, dim=-1)

        # 4. Stop Condition: Se predice [EOS], ci fermiamo
        if predicted_id.item() == tokenizer.token_to_id("[EOS]"):
            break

        # 5. Append: Aggiungiamo il nuovo token all'input per il prossimo giro
        # Concateniamo lungo la dimensione della sequenza (dim=1)
        input_tensor = torch.cat([input_tensor, predicted_id.unsqueeze(0)], dim=1)

        # Controllo di sicurezza: se superiamo la lunghezza massima del modello ( 1024)
        if input_tensor.shape[1] >= model.pos_encoding.pe.shape[1]:
            print("Raggiunta lunghezza massima del modello!")
            break

    # 6. Decodifica
    # Convertiamo gli ID in parole.
    # [0] perché input_tensor ha batch size 1
    full_text = tokenizer.decode(input_tensor[0].tolist(), skip_special_tokens=False)
    print(full_text)
    # Pulizia: Rimuoviamo il prompt originale per vedere solo il riassunto
    summary_only = full_text[full_text.index('[EOS]'):]

    return summary_only

In [44]:
# Ricarico il modello migliroe
checkpoint_path = 'best_transformer_decoder_only.pt'

if torch.cuda.is_available():
    model.load_state_dict(torch.load(checkpoint_path))
else:
    # Se carichiamo su CPU un modello allenato su GPU
    model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))

print("Modello caricato con successo!")

Modello caricato con successo!


In [45]:
nuovo_dialogo = """
Anna: Hey, are you coming to the party tonight?
Mark: I don't think so, I have to study for the exam tomorrow.
Anna: Come on! It will be fun, just for an hour.
Mark: Okay, maybe just for one hour.
"""

# Generiamo!
riassunto = ask_gpt(model, tokenizer, test_dataset.dialogue.iloc[2], device)

print(f"\nGenerato dal modello:\n{riassunto}")

--- Inizio Generazione ---
[BOS] Lenny : Ba be , can you help me with something ? Bob : Sure , what ' s up ? Lenny : Which one should I pick ? Bob : Send me photos Lenny : < file_photo > Lenny : < file_photo > Lenny : < file_photo > Bob : I like the first ones best Lenny : But I already have pur ple tr ous ers . Does it make sense to have two pair s ? Bob : I have four black pair s : D : D Lenny : yeah , but shouldn ' t I pick a different color ? Bob : what matter s is what you ' ll give you the most out fit options Lenny : So I guess I ' ll buy the first or the third pair then Bob : P ick the best quality then Lenny : ur right , thx Bob : no prob :) [EOS] : Lenny wants Lenny to come over to fix her shoes . She will bring his second one . He has a lot of them . com ment . They are going to get one .

Generato dal modello:
[EOS] : Lenny wants Lenny to come over to fix her shoes . She will bring his second one . He has a lot of them . com ment . They are going to get one .
